# Notebook 23 — Pseudo-labeling

**Idée** : le modèle champion (nb08) produit des scores de confiance sur le test.
Pour les transactions test avec score très élevé (quasi-fraude certaine) ou très bas (quasi-légitime certaine),
on peut les ajouter au train comme "pseudo-labels" et ré-entraîner.

**Pourquoi ça peut aider** :
- Le test couvre les périodes 106-143, le train 0-105
- Les patterns de comptes frauduleux **évoluent** : des comptes qui n'apparaissent qu'en test
- Le pseudo-labeling donne au modèle un signal sur ces comptes
- AP est une métrique de ranking → même un pseudo-label imparfait aide si le ranking relatif s'améliore

**Risque** : amplifier les erreurs du modèle initial (biais).
**Mitigation** : seuils très stricts (0.98 / 0.002) + un seul cycle de pseudo-labeling.

**Champion (nb08)** : recent2 ≈ 0.3662, LB 0.3569

**IMPORTANT** : CV classique impossible ici (test n'a pas de labels).
On évalue indirectement via le CV standard sur train, puis on soumet.
Le LB est le seul oracle.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything, make_submission
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv")
test  = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy()
y_all = train[C.TARGET].to_numpy()
folds_full = list(time_folds(train[C.PERIOD]))
print(f"Train: {len(train):,} | Train op03: {op03.sum():,} | Test: {len(test):,}")

In [ ]:
EPS = 1e-6
WINDOWS = (5, 10, 20)
SM = 30

def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]
    f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)

def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref[col].value_counts(normalize=True)
        X[f"freq_{col}"] = df[col].map(freq).fillna(0).values
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    rt  = recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)
    return pd.concat([X, beh, rec, rt], axis=1)

def feats_train(df, ref):
    X = base_build(df, ref)
    X["te_origin"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM)
    return X

def feats_apply(df, ref):
    X = base_build(df, ref)
    mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM)
    X["te_origin"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm)
    return X

def make_cat():
    from catboost import CatBoostClassifier
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=600, random_seed=42, verbose=False)

print("Setup OK")

In [ ]:
# ── ÉTAPE 1 : Entraîner modèle champion sur tout le train op03 ───────────────────
print("Étape 1 : Entraînement champion sur train complet...")
ref_full = train.iloc[np.where(op03)[0]]
yf = y_all[op03]
Xf = feats_train(ref_full, ref_full)
model_step1 = make_cat().fit(Xf, yf)
print(f"  Train op03: {len(ref_full):,} | fraudes: {yf.sum():,} ({100*yf.mean():.1f}%)")

In [ ]:
# ── ÉTAPE 2 : Scorer le test op03 et extraire les pseudo-labels ──────────────────
te_op = op03_mask(test).to_numpy()
test_op03 = test.iloc[np.where(te_op)[0]].copy()
Xte = feats_apply(test_op03, ref_full)
proba_test = model_step1.predict_proba(Xte)[:, 1]

print(f"Test op03: {len(test_op03):,} transactions")
print(f"Distribution des scores:")
for thr in [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]:
    print(f"  proba > {thr}: {(proba_test > thr).sum():,}")

# Seuils stricts pour pseudo-labels
THRESH_FRAUD = 0.98   # quasi-certitude fraude
THRESH_LEGIT = 0.02   # quasi-certitude légitime

pseudo_fraud_mask = proba_test > THRESH_FRAUD
pseudo_legit_mask = proba_test < THRESH_LEGIT

print(f"\nPseudo-labels sélectionnés (seuil fraude={THRESH_FRAUD}, légit={THRESH_LEGIT}):")
print(f"  Pseudo-fraudes: {pseudo_fraud_mask.sum():,}")
print(f"  Pseudo-légitimes: {pseudo_legit_mask.sum():,}")
print(f"  Total pseudo-labels: {(pseudo_fraud_mask | pseudo_legit_mask).sum():,}")

In [ ]:
# ── ÉTAPE 3 : Construire train augmenté ──────────────────────────────────────────
pseudo_mask = pseudo_fraud_mask | pseudo_legit_mask
pseudo_df = test_op03.iloc[np.where(pseudo_mask)[0]].copy()
pseudo_df[C.TARGET] = 0.0  # placeholder
pseudo_df.loc[pseudo_df.index[pseudo_fraud_mask[pseudo_mask]], C.TARGET] = 1.0

# Concat : train op03 + pseudo-labels du test
# IMPORTANT : les pseudo-labels ont des périodes plus récentes (106-143)
# → ils sont naturellement "dans le futur" pour le train
train_aug = pd.concat([ref_full, pseudo_df], ignore_index=True)
y_aug = train_aug[C.TARGET].to_numpy()

print(f"Train augmenté: {len(train_aug):,} ({len(ref_full):,} réels + {len(pseudo_df):,} pseudo)")
print(f"Fraudes: {int(y_aug.sum()):,} dont {int(pseudo_df[C.TARGET].sum()):,} pseudo-fraudes")

In [ ]:
# ── ÉTAPE 4 : Ré-entraîner sur train augmenté ───────────────────────────────────
print("Étape 4 : Ré-entraînement sur train augmenté...")
Xf_aug = feats_train(train_aug, train_aug)
model_step2 = make_cat().fit(Xf_aug, y_aug)

# Scorer le test
Xte2 = feats_apply(test_op03, train_aug)
proba_test2 = model_step2.predict_proba(Xte2)[:, 1]

full = np.zeros(len(test))
full[te_op] = proba_test2

path = make_submission(test[C.ID], full, "23_pseudo_label")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print(f"Soumission: {path} | proba>0: {int((sub['target']>0).sum())}")

In [ ]:
# ── COMPARAISON : Champion vs Pseudo-labeling ────────────────────────────────────
# Corrélation entre les deux prédictions sur le test
corr = np.corrcoef(proba_test, proba_test2)[0,1]
print(f"Corrélation champion vs pseudo: {corr:.4f}")
print(f"  Si ≈1.00 → pseudo-labeling ne change rien")
print(f"  Si <0.99 → changement notable, soumettre")
print()

# Vérifier combien de pseudo-fraudes changent de rang
delta = proba_test2 - proba_test
print(f"Delta moyen (pseudo - champion): {delta.mean():.4f}")
print(f"Delta std: {delta.std():.4f}")
print(f"Max hausse: +{delta.max():.4f} | Max baisse: {delta.min():.4f}")

# Importance features
imp = pd.Series(model_step2.get_feature_importance(), index=Xf_aug.columns).sort_values(ascending=False)
print("\nTop 10 importances (modèle pseudo-labeling):")
print(imp.head(10).to_string())

In [ ]:
# ── VARIANTE : Blend champion + pseudo ──────────────────────────────────────────
# Parfois blend est meilleur que l'un ou l'autre seul
alpha = 0.5  # poids pseudo vs champion
blend = (1 - alpha) * proba_test + alpha * proba_test2
full_blend = np.zeros(len(test))
full_blend[te_op] = blend
path_blend = make_submission(test[C.ID], full_blend, "23_pseudo_blend50")
print(f"Soumission blend 50/50: {path_blend}")
print()
print("=== STRATÉGIE DE SOUMISSION ===")
print(f"  → Soumettre d'abord: 23_pseudo_label (modèle ré-entraîné)")
print(f"  → Si LB amélioré: c'est le nouveau champion")
print(f"  → Si LB baisse: essayer le blend 50/50")
print(f"  → Soumettre max 5 fois par jour")
print(f"  → nb08 (LB 0.3569) reste le fallback")